In [10]:
import math
import gc
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype
from sklearn.decomposition import PCA

# Set random seed for reproducibility
np.random.seed(42)

# Import custom libraries (adjust paths as needed)
from lib.aggregate.cell_data_utils import (
    load_metadata_cols, 
    split_cell_data,
    get_feature_table_cols
)
from lib.aggregate.align import (
    prepare_alignment_data,
    centerscale_by_batch,
    centerscale_on_controls,
    tvn_on_controls,
)
from lib.aggregate.aggregate import aggregate

print("✓ Imports complete")

✓ Imports complete


In [19]:
# ========== CONFIGURE THESE PATHS ==========

# Input: Your filtered parquet files
# Can be a single file or a list/glob pattern
filtered_paths = [
    "/lab/ops_analysis/lourido/nebo-analysis/analysis/analysis_root/aggregate/parquets/P-1_W-A3_CeCl-all_ChCo-DAPI_Stat6_Tubulin_WGA_cMYC_CoCo-nucleus__filtered.parquet"
]

# Output paths
output_dir = Path("/lab/ops_analysis/lourido/nebo-analysis/output_analysis/agg_output")
output_dir.mkdir(exist_ok=True, parents=True)

feature_table_output = output_dir / "feature_table.tsv"
aligned_output = output_dir / "aligned.parquet"
aggregated_output = output_dir / "aggregated.tsv"

# Configuration parameters
metadata_cols_fp = "/lab/ops_analysis/lourido/nebo-analysis/analysis/config/cell_data_metadata_cols.tsv"  # File listing metadata column names
perturbation_name_col = "gene_symbol_0"  # Column with gene/perturbation names
perturbation_id_col = "gene_symbol_0"  # Column with sgRNA IDs
control_key = "nontargeting_"  # Prefix for control perturbations
batch_cols = ["plate"]  # Columns defining batches
variance_or_ncomp = 0.95  # PCA: keep components explaining 95% variance (or int for n components)
num_align_batches = 10  # Number of batches for processing alignment
agg_method = "median"  # Aggregation method: "median" or "mean"

# Optional: vacuole columns to include in feature table
vacuole_cols = [
    "num_vacuoles",
    "total_vacuole_area", 
    "vacuole_area_ratio",
    "mean_vacuole_diameter",
    "mean_distance_to_nucleus",
]

print("✓ Configuration complete")

✓ Configuration complete


In [20]:
# ========== STEP 3: GENERATE FEATURE TABLE ==========
print("\n" + "="*60)
print("STEP 3: Generating Feature Table")
print("="*60)

# Load cell data using PyArrow dataset
print("\nLoading cell data...")
cell_data = ds.dataset(filtered_paths, format="parquet")

# Determine columns
cell_data_cols = cell_data.schema.names
# Don't include classification cols since we're working with filtered data directly
metadata_cols = load_metadata_cols(metadata_cols_fp, include_classification_cols=False)
feature_cols = [col for col in cell_data_cols if col not in metadata_cols]
feature_cols = get_feature_table_cols(feature_cols)

# Add vacuole columns if they exist
available_vacuole_cols = [col for col in vacuole_cols if col in cell_data_cols]
if available_vacuole_cols:
    feature_cols.extend(available_vacuole_cols)
    print(f"Added vacuole columns: {available_vacuole_cols}")

# Load data and convert to float32
cell_data = cell_data.to_table(
    columns=metadata_cols + feature_cols, 
    use_threads=True
).to_pandas()

print(f"Shape of input data: {cell_data.shape}")

# Convert numeric columns to float32
for col in cell_data.columns:
    if is_numeric_dtype(cell_data[col]):
        cell_data[col] = cell_data[col].astype("float32")

# Split metadata and features
metadata, features = split_cell_data(cell_data, metadata_cols)
del cell_data
gc.collect()

# Create batch_values column manually
if len(batch_cols) > 0:
    metadata['batch_values'] = metadata[batch_cols].astype(str).agg('_'.join, axis=1)
else:
    metadata['batch_values'] = 'batch_0'

# Filter to only include valid perturbations
valid_mask = (
    metadata[perturbation_name_col].notna() &
    (metadata[perturbation_name_col] != '')
)
metadata = metadata[valid_mask].copy()
features = features.loc[metadata.index].copy()

features = features.astype(np.float32)

# Modify perturbation column for non-targeting controls (append sgRNA ID)
control_ind = metadata[perturbation_name_col].str.startswith(control_key).to_list()
metadata.loc[control_ind, perturbation_name_col] = (
    metadata.loc[control_ind, perturbation_name_col]
    + "_"
    + metadata.loc[control_ind, perturbation_id_col]
)

# Center-scale features on controls
print("Center-scaling features on controls...")
features = centerscale_on_controls(
    features,
    metadata,
    perturbation_name_col,
    control_key,
    "batch_values",
).astype(np.float32)

# Aggregate by taking median per perturbation
print("Aggregating features by perturbation...")
features = pd.DataFrame(features, columns=feature_cols)
features[perturbation_name_col] = metadata[perturbation_name_col].values
features = features.groupby(perturbation_name_col, sort=False, observed=True).median()
features = features.reset_index()

# Save feature table
features.to_csv(feature_table_output, sep="\t", index=False)
print(f"\n✓ Feature table saved to: {feature_table_output}")
print(f"  Shape: {features.shape}")
print(f"  Perturbations: {len(features)}")


STEP 3: Generating Feature Table

Loading cell data...
Added vacuole columns: ['num_vacuoles', 'total_vacuole_area', 'vacuole_area_ratio']
Shape of input data: (4914, 119)
Center-scaling features on controls...
Aggregating features by perturbation...

✓ Feature table saved to: /lab/ops_analysis/lourido/nebo-analysis/output_analysis/agg_output/feature_table.tsv
  Shape: (49, 64)
  Perturbations: 49


/lab/ops_analysis/lourido/nebo-analysis/brieflow/workflow/lib/aggregate/align.py:234: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  batch_ind & (metadata[pert_col].str.startswith(control_key)).to_list()


In [21]:
# ========== STEP 4: ALIGN DATA ==========
print("\n" + "="*60)
print("STEP 4: Aligning Data (PCA + Batch Correction)")
print("="*60)

PCA_SUBSET = 100000

# Load full dataset
cell_dataset = ds.dataset(filtered_paths, format="parquet")
total_rows = cell_dataset.count_rows()
print(f"\nTotal rows across all files: {total_rows:,}")

# --- Step 4a: Fit PCA on subset ---
print(f"\nFitting PCA on random subset of {min(PCA_SUBSET, total_rows):,} cells...")
n_sample = min(PCA_SUBSET, total_rows)
random_indices = np.random.choice(total_rows, size=n_sample, replace=False)
random_indices.sort()

# Load sample
sample_df = cell_dataset.scanner().take(random_indices)
sample_df = sample_df.to_pandas(use_threads=True, memory_pool=None)

# Prepare sample for PCA
metadata_cols_list = load_metadata_cols(metadata_cols_fp, include_classification_cols=False)
metadata_sample, features_sample = split_cell_data(sample_df, metadata_cols_list)

# Prepare alignment data - this adds batch_values column and filters invalid data
metadata_sample, features_sample = prepare_alignment_data(
    metadata_sample,
    features_sample,
    batch_cols,
)

# Fit PCA
pca = PCA(n_components=variance_or_ncomp).fit(
    centerscale_by_batch(features_sample, metadata_sample, "batch_values")
)
print(f"✓ PCA fitted with {pca.n_components_} components")
print(f"  Explained variance: {pca.explained_variance_ratio_.sum():.2%}")

del sample_df, metadata_sample, features_sample
gc.collect()

# --- Step 4b: Process in batches ---
print(f"\nProcessing data in {num_align_batches} batches...")

# Create batch indices
all_indices = np.random.permutation(total_rows)
chunk_size = math.ceil(total_rows / num_align_batches)
subset_indices = [
    all_indices[i * chunk_size : (i + 1) * chunk_size] 
    for i in range(num_align_batches)
]

# Process each batch
writer = None
for i, indices in enumerate(subset_indices):
    print(f"  Batch {i + 1}/{num_align_batches}: {len(indices):,} cells")
    
    # Load batch
    subset_df = cell_dataset.scanner().take(pa.array(indices))
    subset_df = subset_df.to_pandas(use_threads=True, memory_pool=None)
    subset_df = subset_df.dropna(axis=1)
    
    # Convert to float32
    for col in subset_df.columns:
        if is_numeric_dtype(subset_df[col]):
            subset_df[col] = subset_df[col].astype("float32")
    
    # Split and prepare
    metadata, features = split_cell_data(subset_df, metadata_cols_list)
    del subset_df
    gc.collect()
    
    # Prepare alignment data - this adds batch_values column
    metadata, features = prepare_alignment_data(
        metadata,
        features,
        batch_cols,
    )
    
    # Center-scale by batch
    features = centerscale_by_batch(features, metadata, "batch_values")
    
    # Transform with PCA
    features = pca.transform(features)
    
    # Apply TVN on controls
    features = tvn_on_controls(
        features,
        metadata,
        perturbation_id_col,
        control_key,
        "batch_values",
    )
    
    # Combine with metadata
    feature_columns = [f"PC_{j}" for j in range(features.shape[1])]
    features = pd.DataFrame(features, index=metadata.index, columns=feature_columns)
    aligned_cell_data = pd.concat([metadata, features], axis=1)
    del features
    gc.collect()
    
    # Write to parquet (streaming)
    aligned_cell_data = pa.Table.from_pandas(aligned_cell_data, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(aligned_output, aligned_cell_data.schema)
    writer.write_table(aligned_cell_data)

# Close writer
if writer is not None:
    writer.close()

print(f"\n✓ Aligned data saved to: {aligned_output}")
print(f"  Components: {pca.n_components_}")


STEP 4: Aligning Data (PCA + Batch Correction)

Total rows across all files: 4,914

Fitting PCA on random subset of 4,914 cells...
✓ PCA fitted with 137 components
  Explained variance: 95.06%

Processing data in 10 batches...
  Batch 1/10: 492 cells


/lab/ops_analysis/lourido/nebo-analysis/brieflow/workflow/lib/aggregate/align.py:234: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  batch_ind & (metadata[pert_col].str.startswith(control_key)).to_list()
/lab/ops_analysis/lourido/nebo-analysis/brieflow/workflow/lib/aggregate/align.py:164: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  batch_ind & (metadata[pert_col].str.startswith(control_key)).to_list()


  Batch 2/10: 492 cells


/lab/ops_analysis/lourido/nebo-analysis/brieflow/workflow/lib/aggregate/align.py:234: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  batch_ind & (metadata[pert_col].str.startswith(control_key)).to_list()
/lab/ops_analysis/lourido/nebo-analysis/brieflow/workflow/lib/aggregate/align.py:164: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  batch_ind & (metadata[pert_col].str.startswith(control_key)).to_list()


ValueError: Table schema does not match schema used to create file: 
table:
plate: float
well: string
tile: float
cell_0: float
i_0: float
j_0: float
site: float
cell_1: float
i_1: float
j_1: float
distance: float
fov_distance_0: float
fov_distance_1: float
gene_symbol_0: string
mapped_single_gene: float
channels_min: float
nucleus_i: float
nucleus_j: float
nucleus_bounds_0: float
nucleus_bounds_1: float
nucleus_bounds_2: float
nucleus_bounds_3: float
cell_i: float
cell_j: float
cell_bounds_0: float
cell_bounds_1: float
cell_bounds_2: float
cell_bounds_3: float
cytoplasm_i: float
cytoplasm_j: float
cytoplasm_bounds_0: float
cytoplasm_bounds_1: float
cytoplasm_bounds_2: float
cytoplasm_bounds_3: float
stitched_cell_id_0: float
stitched_cell_id_1: float
cell_barcode_0: string
has_vacuole: float
num_vacuoles: float
vacuole_ids: string
vacuole_id: float
nearest_nucleus_id: float
overlap_ratio: float
vacuole_i: float
vacuole_j: float
vacuole_bounds_0: float
vacuole_bounds_1: float
vacuole_b

In [22]:
# Add this debug cell before Step 4 to check your data:
# ========== DEBUG: Check Control Samples ==========

print("\n" + "="*60)
print("DEBUG: Checking Control Samples")
print("="*60)

# Load a small sample to inspect
test_data = ds.dataset(filtered_paths, format="parquet")
test_df = test_data.to_table(use_threads=True).to_pandas()

print(f"\nDataset shape: {test_df.shape}")
print(f"\nColumns in data: {test_df.columns.tolist()}")

# Check perturbation columns
if perturbation_name_col in test_df.columns:
    print(f"\nUnique values in {perturbation_name_col}:")
    print(test_df[perturbation_name_col].value_counts().head(20))
else:
    print(f"\n⚠️  Column '{perturbation_name_col}' not found in data!")

if perturbation_id_col in test_df.columns:
    print(f"\nUnique values in {perturbation_id_col}:")
    print(test_df[perturbation_id_col].value_counts().head(20))
else:
    print(f"\n⚠️  Column '{perturbation_id_col}' not found in data!")

# Check for controls
print(f"\nSearching for controls with key: '{control_key}'")
if perturbation_name_col in test_df.columns:
    controls_in_name = test_df[perturbation_name_col].str.startswith(control_key, na=False).sum()
    print(f"Controls found in {perturbation_name_col}: {controls_in_name}")
    
if perturbation_id_col in test_df.columns:
    controls_in_id = test_df[perturbation_id_col].str.startswith(control_key, na=False).sum()
    print(f"Controls found in {perturbation_id_col}: {controls_in_id}")

# Check batch columns
print(f"\nBatch columns: {batch_cols}")
for col in batch_cols:
    if col in test_df.columns:
        print(f"  {col}: {test_df[col].nunique()} unique values")
    else:
        print(f"  ⚠️  {col}: NOT FOUND")


DEBUG: Checking Control Samples

Dataset shape: (4914, 724)

Columns in data: ['plate', 'well', 'tile', 'cell_0', 'i_0', 'j_0', 'site', 'cell_1', 'i_1', 'j_1', 'distance', 'fov_distance_0', 'fov_distance_1', 'gene_symbol_0', 'mapped_single_gene', 'channels_min', 'nucleus_i', 'nucleus_j', 'nucleus_bounds_0', 'nucleus_bounds_1', 'nucleus_bounds_2', 'nucleus_bounds_3', 'cell_i', 'cell_j', 'cell_bounds_0', 'cell_bounds_1', 'cell_bounds_2', 'cell_bounds_3', 'cytoplasm_i', 'cytoplasm_j', 'cytoplasm_bounds_0', 'cytoplasm_bounds_1', 'cytoplasm_bounds_2', 'cytoplasm_bounds_3', 'stitched_cell_id_0', 'stitched_cell_id_1', 'cell_barcode_0', 'has_vacuole', 'num_vacuoles', 'vacuole_ids', 'vacuole_id', 'nearest_nucleus_id', 'overlap_ratio', 'vacuole_i', 'vacuole_j', 'vacuole_bounds_0', 'vacuole_bounds_1', 'vacuole_bounds_2', 'vacuole_bounds_3', 'vacuole_number_neighbors_1', 'vacuole_percent_touching_1', 'vacuole_first_neighbor_distance', 'vacuole_second_neighbor_distance', 'vacuole_angle_between_nei

In [ ]:
# ========== STEP 5: AGGREGATE ==========
print("\n" + "="*60)
print("STEP 5: Aggregating to Perturbation Level")
print("="*60)

# Load aligned cell data
print("\nLoading aligned data...")
cell_data = ds.dataset(aligned_output, format="parquet")
cell_data = cell_data.to_table(use_threads=True, memory_pool=None).to_pandas()
print(f"Shape of aligned data: {cell_data.shape}")

# Split into metadata and features
metadata_cols_list = load_metadata_cols(
    metadata_cols_fp, 
    include_classification_cols=True
) + ["batch_values"]

metadata, tvn_normalized = split_cell_data(cell_data, metadata_cols_list)
tvn_normalized = tvn_normalized.to_numpy()
del cell_data
gc.collect()

# Aggregate
print(f"Aggregating using method: {agg_method}")
aggregated_embeddings, aggregated_metadata = aggregate(
    tvn_normalized,
    metadata,
    perturbation_name_col,
    agg_method,
)

# Create output dataframe
feature_columns = [f"PC_{i}" for i in range(tvn_normalized.shape[1])]
aggregated_embeddings_df = pd.DataFrame(
    aggregated_embeddings, 
    index=aggregated_metadata.index, 
    columns=feature_columns
)

aggregated_cell_data = (
    pd.concat([aggregated_metadata, aggregated_embeddings_df], axis=1)
    .sort_values("cell_count", ascending=False)
    .reset_index(drop=True)
)

# Save
aggregated_cell_data.to_csv(aggregated_output, sep="\t", index=False)
print(f"\n✓ Aggregated data saved to: {aggregated_output}")
print(f"  Shape: {aggregated_cell_data.shape}")
print(f"  Perturbations: {len(aggregated_cell_data)}")
print(f"\nTop perturbations by cell count:")
print(aggregated_cell_data[[perturbation_name_col, "cell_count"]].head(10))

In [ ]:
# ========== SUMMARY ==========
print("\n" + "="*60)
print("PIPELINE COMPLETE!")
print("="*60)

print(f"""
Output Files Generated:
1. Feature Table: {feature_table_output}
   - Per-perturbation median features (raw features)
   - Shape: {features.shape if 'features' in dir() else 'N/A'}

2. Aligned Data: {aligned_output}
   - Single-cell data with PCA transformation and batch correction
   - Components: {pca.n_components_ if 'pca' in dir() else 'N/A'}

3. Aggregated Data: {aggregated_output}
   - Perturbation-level aggregated embeddings
   - Shape: {aggregated_cell_data.shape if 'aggregated_cell_data' in dir() else 'N/A'}

Next Steps:
- Use aggregated_output for perturbation-level analysis
- Use feature_table_output for feature interpretation
- Use aligned_output for single-cell analysis
""")